# Notebook for Training the MacroTrader Models

In [ ]:
# installing packages for you
# !pip install --upgrade setuptools packaging
# !pip install -e ..

In [ ]:
# Install TA-Lib using conda
# !conda install -c conda-forge ta-lib -y

In [ ]:
!nvidia-smi

In [ ]:
from blockhouse_ml.options.utils.data_handler import DataHandler
from blockhouse_ml.options.utils.macro_model_utils import MacroTraderModel
from blockhouse_ml.utils.macro_model import MetaLearner

In [ ]:
import os
import pandas as pd
import numpy as np
import time
import torch
from stable_baselines3 import PPO
from gym import spaces
from collections import defaultdict
from tqdm import tqdm

In [ ]:
# # Initialize the model directory, where the models will be saved
TAB_MODEL_DIR = '../OptionsTabModels_v3'
os.makedirs(TAB_MODEL_DIR, exist_ok=True)

# Initialize the log
log_dir = 'Logs'

# Initialize the MetaLearner
metalearner = MetaLearner()

macro_trader_tab = MacroTraderModel(TAB_MODEL_DIR, num_envs=1)

## Model Training parameters
training_params = { 'callback' : None, 'total_timesteps' : 10000}

# Initialize the data directory, where the data will be stored
data_dir = '../OptionsData1'
os.makedirs(data_dir, exist_ok=True)

data_handler = DataHandler(data_dir=data_dir)

# Define the start and end dates (15 days before maturity)
start_time = '2024-07-08'
end_time = '2024-08-22'

# List of Companies based on Market Capitalization
large_cap_companies = [
    {"ticker": 'AAPL',"option_type": "C","strike_price": 100,"maturity_date": "240823"},
    {"ticker" : 'CSCO',"option_type": "C","strike_price": 53,"maturity_date": "240823"}, 
    {"ticker": 'MSFT',"option_type": "C","strike_price": 420,"maturity_date": "240823"},
    {'ticker': 'MCD', 'option_type': 'C', 'strike_price': 200, 'maturity_date': '240823'},
    {'ticker': 'AMZN', 'option_type': 'C', 'strike_price': 100, 'maturity_date': '240823'},
    {'ticker': 'TSLA', 'option_type': 'C', 'strike_price': 100, 'maturity_date': '240823'},
    {'ticker': 'PFE', 'option_type': 'C', 'strike_price': 30, 'maturity_date': '240823'},
    {'ticker': 'NVDA', 'option_type': 'C', 'strike_price': 100, 'maturity_date': '240823'},
    {'ticker': 'IBM', 'option_type': 'C', 'strike_price': 100, 'maturity_date': '240823'},
    {'ticker': 'MS', 'option_type': 'C', 'strike_price': 100, 'maturity_date': '240823'}]#['AAPL', 'CSCO', 'MCD', 'IBM', 'AMZN', 'TSLA', 'PFE', 'MS','MSFT','NVDA']
mid_cap_companies = []#['AEG', 'NICE', 'NLY', 'ONTO', 'PSN', 'SAIA', 'OWL','PNW','TWLO','HAS']
small_cap_companies = []#['NVAX','AMC','WOLF','IREN','SEDG', 'UPWK','SERV','FSLY','BMBL','ARRY']


# Fetch and process the data for Training

In [ ]:
# # Fetch and process the data and save it to the data directory
# Fetch and process the data and save it to the data directory
for companies in [large_cap_companies, mid_cap_companies, small_cap_companies]:
    for option_data in companies:
        print("------------------------------------------------")
        print(f"Fetching Option Data for {option_data['ticker']}")
        processed_data = data_handler.get_data(option_data, start_time, end_time)

        

In [ ]:
# import pytz
# processed_path = f"{data_dir}/processed-option-train-data.csv"
# ticker= 'AAPL'
# ## User specific Option data
# user_data = {
#     "option_type" : "C",
#     "strike_price" : 100,
#     "maturity_date" : "240823"
# }

# if os.path.exists(processed_path):
#     processed_data = pd.read_csv(processed_path)
# else:
#     quicker_training_filepath = f'{data_dir}/options-train-data.csv'
#     output_path = f'{data_dir}/macro-trader-training-data.csv'


#     est_tz = pytz.timezone('America/New_York')

#     processed_data = pd.read_csv(quicker_training_filepath)
#     processed_data['contract'] = f"{ticker}  {user_data['maturity_date']}C00100000"
#     processed_data['datetime'] = pd.to_datetime(processed_data['timestamp'])
#     # processed_data['timestamp'] = processed_data['datetime']
#     processed_data['VWAP'] = processed_data['VWAP_bid']
#     processed_data = processed_data[processed_data['datetime'].dt.dayofweek < 5]
#     processed_data.set_index('datetime', inplace=True)
#     # print(processed_data)
#     processed_data = processed_data.between_time('13:30', '20:00')

#     processed_data.reset_index(inplace=True)
#     processed_data =data_processor.process_data(processed_data,option_type=user_data['option_type'],strike_price=user_data['strike_price'],n_jobs=2)
#     # processed_data['datetime'] = processed_data['datetime']
#     print(processed_data.columns)
#     processed_data.to_csv(output_path)
# processed_data

# Training the Unet-Transformer MacroTrader Model without Fine Tuning 

In [ ]:
# # from collections import defaultdict
# results = defaultdict(list)

# # for companies in [large_cap_companies, mid_cap_companies, small_cap_companies]:

# #     for ticker in companies:

# #         processed_data = get_data(data_dir, ticker, start_time, end_time)
        

# train_data = processed_data[:int(len(processed_data) * 0.8)]
# test_data = processed_data[int(len(processed_data) * 0.8):]

# print (" Training Data Shape: ", train_data.shape)

# print("Training Model for ", ticker)
# market_cap_int = data_client.get_market_cap(ticker)
# market_cap = metalearner.classify_market_cap(market_cap_int)
# for transaction_size in [10000]:
#     scenerio = metalearner.classify_scenario(transaction_size)
#     print("Training Model for ticker: {} Transaction Size: {} Market Cap: {}".format(ticker, scenerio, market_cap))
#     training_params['tb_log_name'] =f'{log_dir}/{ticker}_{scenerio}_{market_cap}'
#     start_t = time.perf_counter()
#     model, _ = macro_trader_unet.train(train_data, market_cap=market_cap,scenario=scenerio,resume=True,training_params = training_params)
#     print("Model Training Time: ", time.perf_counter() - start_t)
#     print("------------------------")
#     print("Testing Model for ticker: {} Transaction Size: {} Market Cap: {}".format(ticker, scenerio, market_cap))
#     rew, _ = macro_trader_unet.test(test_data, model=model, market_cap=market_cap, scenario=scenerio)
#     results[ticker].append((market_cap, scenerio, rew))
#     print("Reward: ", rew)
#     print("------------------------")


# Training the TAB-Transformer MacroTrader Model without Fine Tuning 

In [ ]:

results = defaultdict(list)
# Fetch and process the data and save it to the data directory
for companies in [large_cap_companies, mid_cap_companies, small_cap_companies]:
    for option_data in companies:
        print("------------------------------------------------")
        print(f"Fetching Option Data for {option_data['ticker']}")
        processed_data = data_handler.get_data(option_data, start_time, end_time)
        
        train_data = processed_data[:int(len(processed_data) * 0.8)]
        test_data = processed_data[int(len(processed_data) * 0.8):]

        # print (" Training Data Shape: ", train_data.shape)
        ticker = option_data['ticker']
        print("Training Model for ", ticker)
        market_cap_int = data_handler.get_market_cap(ticker)
        market_cap = metalearner.classify_market_cap(market_cap_int)
        for transaction_size in [9, 99, 499, 1999, 10000]:
            scenerio = metalearner.classify_scenario(transaction_size)
            print("Training Model for ticker: {} Transaction Size: {} Market Cap: {}".format(ticker, scenerio, market_cap))
            training_params['tb_log_name'] =f'{log_dir}/{ticker}_{scenerio}_{market_cap}'
            start_t = time.perf_counter()
            model, _ = macro_trader_tab.train(train_data, market_cap=market_cap,scenario=scenerio,resume=True,training_params = training_params, get_tab_transformer=True)
            print("Model Training Time: ", time.perf_counter() - start_t)
            print("------------------------")
            print("Testing Model for ticker: {} Transaction Size: {} Market Cap: {}".format(ticker, scenerio, market_cap))
            rew, _, _ = macro_trader_tab.test(test_data, model=model, market_cap=market_cap, scenario=scenerio)
            results[ticker].append((market_cap, scenerio, rew))
            print("Reward: ", rew)
            print("------------------------")


# Training with finetuning

In [ ]:
# import ray
# cwd = os.getcwd()
# print(cwd)
# ray.init(local_mode=True, num_gpus=1, num_cpus=16, dashboard_host='0.0.0.0', log_to_driver=False,
#              _temp_dir=os.path.join(cwd, "ray_results"), ignore_reinit_error=True)


# Declare the variables for finetuning it can be different then training the model above

In [ ]:
# # Initialize the model directory, where the models will be saved
# MODEL_DIR = 'Fine-Tuned-Models'
# os.makedirs(MODEL_DIR, exist_ok=True)

# # Initialize the data directory, where the data will be stored
# data_dir = 'Data'
# os.makedirs(data_dir, exist_ok=True)

# # Initialize the log
# log_dir = 'Logs'

# # Initialize the MetaLearner
# metalearner = MetaLearner()

# # Initialize the MacroTraderModel
# macro_trader_unet = MacroTraderModel(MODEL_DIR)

# # Initialize the data processor
# data_processor = DataProcessor()

# ## Model Training parameters
# training_params = { 'callback' : None, 'total_timesteps' : 10000}

# # Define the forecast steps
# forecast_steps = {
#     'open': (360, '1T'),
#     'high': (360, '1T'),
#     'low': (360, '1T'),
#     'close': (360, '1T'),
#     'volatility': (360, '1T'),
#     'volume': (360, '1T'),
#     'transaction_cost': (360, '1T')
# }

# # Define the start and end dates
# start_time = '2024-07-01'
# end_time = '2024-08-16'

# # List of Companies based on Market Capitalization, for fine tuning we only use 1 company for each market cap
# large_cap_companies = ['AAPL']
# mid_cap_companies = ['AEG']
# small_cap_companies = ['NVAX']


In [ ]:
# from ray import tune, train
# from ray.tune.schedulers import ASHAScheduler
# from ray.tune.search.hyperopt import HyperOptSearch
# import ray


# storage_path = "~/tune_results"

# # Training Config
# """
# Key required for fine tuning
# learning_rate, n_steps, batch_size, gamma, clip_range, n_epochs, ent_coef, resume, total_timesteps, callback, tb_log_name, train_data, market_cap, scenario, test_data
# """

# finetune_config = {
#     "learning_rate": tune.loguniform(1e-5, 1e-1),
#     "n_steps": tune.choice([128, 256, 512]),
#     "batch_size": tune.choice([64, 128, 256]),
#     "gamma": tune.uniform(0.9, 0.999),
#     "clip_range" : tune.uniform(0.1, 0.4),
#     "n_epochs": tune.choice([4, 6, 8]),
#     "ent_coef": tune.loguniform(0.0001, 0.1),
#     "resume" : False,
#     "total_timesteps": training_params['total_timesteps'],
#     "callback" : training_params['callback']
# }

# scheduler = ASHAScheduler(
#     metric="reward",
#     mode="max",
# )

# # hyperopt_search = HyperOptSearch(finetune_config, metric="reward", mode="max")


# for companies in [large_cap_companies, mid_cap_companies, small_cap_companies]:

#     for ticker in companies:
#         filename = f'{data_dir}/merged_data_{ticker}_{start_time}_{end_time}.csv'
#         if os.path.exists(filename):
#             data = pd.read_csv(filename)
#         else:
#             data = fetch_merge_data.fetch_and_merge_data(ticker,start_date=start_time,end_date=end_time,save_dir =data_dir)

#         processed_filename = f'{data_dir}/processed_data_{ticker}_{start_time}_{end_time}.csv'

#         if os.path.exists(processed_filename):
#             processed_data = pd.read_csv(processed_filename)
#         else:
#             processed_data = data_processor.process_data(data, forecast_steps)
#             processed_data.to_csv(processed_filename)
#         train_data = processed_data[:int(len(processed_data) * 0.8)]
#         test_data = processed_data[int(len(processed_data) * 0.8):]

#         # print (" Training Data Shape: ", train_data.shape)

#         # print("Training Model for ", ticker)
#         market_cap_int = fetch_merge_data.get_market_cap(ticker)
#         market_cap = metalearner.classify_market_cap(market_cap_int)
#         for transaction_size in [9]:#, 99, 499, 1999, 10000]:
#             scenerio = metalearner.classify_scenario(transaction_size)

#             finetune_config["train_data"] = train_data
#             finetune_config["test_data"] = test_data
#             finetune_config["market_cap"] = market_cap
#             finetune_config["scenario"] = scenerio
            
#             analysis = tune.Tuner(
#                 macro_trader_unet.fine_tuning_model,
#                 param_space=finetune_config,
#                 run_config=train.RunConfig(
#                     name=f"{ticker}_{scenerio}_{market_cap}",
#                     storage_path="~/tune_results",
#                 ),
#                 tune_config=tune.TuneConfig(
#                     scheduler=scheduler,
#                      trial_dirname_creator=lambda trial: trial.trial_id)
#             )

#             results = analysis.fit()
#             print(results.metrics_dataframe())
#             break
#         break

# # print("Best config: ", analysis.get_best_config(metric="mean_reward", mode="max"))
# # df = analysis.results_df
# # print(df.head())

In [ ]:
# dfs = {result.path: result.metrics_dataframe for result in results}
# [d.mean_accuracy.plot() for d in dfs.values()]